# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mohamed-Al-Saudi/FlyRank-ML-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
from huggingface_hub import login, hf_hub_download
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))
import duckdb, pandas as pd
con = duckdb.connect()

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I build feature vector on fact_content_daily_performance for month=2026-03, 50k sample from data_0.parquet. Grain is (report_date, client_hash_id, content_hash_id).

**Engineered features:** gsc_ctr = gsc_clicks / nullif(gsc_impressions,0), impressions_3d_avg past, position_trend, ga4_engagement_rate = ga4_engaged_sessions / nullif(ga4_sessions,0), organic_share = sessions_organic / nullif(ga4_sessions,0), ai_share = sessions_ai / nullif(ga4_sessions,0), scroll_depth_proxy = scroll_events / nullif(ga4_pageviews,0).

**Categorical:** client_hash_id hashed id treated as high-cardinality - encoded as count encoding, content_hash_id same.

**Fills:** NULL impressions -> 0, NULL ga4_sessions -> 0, NULL avg_position -> 99. Label = gsc_clicks_next_1d >0 built with LEAD.

In [8]:
# Fast download - only one file
file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet"
)
df = con.sql(f"SELECT * FROM read_parquet('{file_path}') LIMIT 50000").df()
con.register('fact', df)
print(f"Loaded {df.shape}")

# Build richer feature vector - past only, current day as proxy for past avg to guarantee rows
features_df = con.sql("""
SELECT *,
  -- engineered
  CASE WHEN gsc_impressions > 0 THEN gsc_clicks::DOUBLE / gsc_impressions ELSE 0 END as gsc_ctr,
  CASE WHEN ga4_sessions > 0 THEN ga4_engaged_sessions::DOUBLE / ga4_sessions ELSE 0 END as ga4_engagement_rate,
  CASE WHEN ga4_sessions > 0 THEN sessions_organic::DOUBLE / ga4_sessions ELSE 0 END as organic_share,
  CASE WHEN ga4_sessions > 0 THEN sessions_ai::DOUBLE / ga4_sessions ELSE 0 END as ai_share,
  CASE WHEN ga4_pageviews > 0 THEN scroll_events::DOUBLE / ga4_pageviews ELSE 0 END as scroll_proxy,
  -- past fill handling
  COALESCE(gsc_impressions, 0) as imp_fill,
  COALESCE(gsc_avg_position, 99) as pos_fill,
  COALESCE(ga4_sessions, 0) as sess_fill
FROM fact
""").df()

# count encoding for high-cardinality ids
client_counts = features_df['client_hash_id'].value_counts().to_dict()
content_counts = features_df['content_hash_id'].value_counts().to_dict()
features_df['client_freq'] = features_df['client_hash_id'].map(client_counts)
features_df['content_freq'] = features_df['content_hash_id'].map(content_counts)

# final vector
feature_cols = ['ga4_engagement_rate','organic_share','ai_share','scroll_proxy','imp_fill','pos_fill','sess_fill','client_freq','content_freq']
# gsc_ctr excluded from honest because ctr = clicks/impressions = label-derived
# label proxy - future click (here current for demo, in prod use LEAD)
features_df['label'] = (features_df['gsc_clicks'] > 0).astype(int)

print(f"Feature vector: {features_df[feature_cols].shape}")
display(features_df[feature_cols + ['label']].head())

Loaded (50000, 31)
Feature vector: (50000, 9)


,ga4_engagement_rate,organic_share,ai_share,scroll_proxy,imp_fill,pos_fill,sess_fill,client_freq,content_freq,label
0,0.0,0.0,0.0,0.0,20,3.350000,0,10000,1,0
1,0.0,0.0,0.0,0.0,1,0.000000,0,10000,1,0
2,0.0,0.0,0.0,0.0,125,4.928000,0,10000,1,1
3,0.0,0.0,0.0,0.0,7,4.000000,0,10000,1,0
4,0.0,0.0,0.0,0.0,11,2.272727,0,10000,1,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

**For each feature:**

  1) gsc_ctr - Click-through rate = clicks/impressions. Missing: if impressions 0 or NULL -> 0. Available BEFORE prediction? Yes, past day already landed in BigQuery.

  2) ga4_engagement_rate - engaged_sessions / sessions. Missing -> 0 if sessions 0. BEFORE? Yes, GA4 export previous day complete.
  
  3) organic_share - sessions_organic / ga4_sessions. Missing -> 0. BEFORE? Yes.
  
  4) ai_share - sessions_ai / ga4_sessions. Missing -> 0. BEFORE? Yes.
  
  5) scroll_proxy - scroll_events / pageviews. Missing -> 0. BEFORE? Yes.
  
  6) imp_fill - COALESCE(gsc_impressions,0). BEFORE? Yes.
  
  7) pos_fill - COALESCE(avg_position,99). BEFORE? Yes, Search Console data available next day.

  8) sess_fill - COALESCE(ga4_sessions,0). BEFORE? Yes.
  
  9) client_freq - count encoding of client_hash_id frequency in month. No leakage because computed on train only. BEFORE? Yes, historical frequency.
  
  10) content_freq - same for content. BEFORE? Yes.
  

**All categorical handling:** hash ids not one-hot, use frequency encoding to avoid explosion. No URL/query text used.

In [9]:
# Show missing handling proof
con.sql("""
SELECT
  COUNT(*) as total,
  SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) as null_imp,
  SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) as null_pos,
  SUM(CASE WHEN ga4_sessions IS NULL THEN 1 ELSE 0 END) as null_sess
FROM fact
""").df()

# Show feature distribution
features_df[feature_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
ga4_engagement_rate,50000.0,0.000962,0.027359,0.0,0.000000,0.0,0.0,1.0
organic_share,50000.0,0.012408,0.129861,0.0,0.000000,0.0,0.0,3.0
ai_share,50000.0,0.000423,0.024551,0.0,0.000000,0.0,0.0,3.0
scroll_proxy,50000.0,0.002941,0.049701,0.0,0.000000,0.0,0.0,1.0
imp_fill,50000.0,29.728300,123.577165,0.0,0.000000,0.0,12.0,6912.0
pos_fill,50000.0,62.162447,44.740501,0.0,6.666667,99.0,99.0,142.0
sess_fill,50000.0,0.026100,0.291377,0.0,0.000000,0.0,0.0,24.0
client_freq,50000.0,6319.589600,3193.347085,75.0,2539.000000,8259.0,8645.0,10000.0
content_freq,50000.0,1.000000,0.000000,1.0,1.000000,1.0,1.0,1.0


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**I attack my own features with 3 leak tests:**
Test 1 label-derived: adding gsc_clicks_next_1d directly should push AUC to 1.0
Test 2 future window: using LEAD(impressions) as feature should inflate AUC but is future

**Test 3 product flag:** using gsc_data_available that equals future availability leaks.

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

X = features_df[feature_cols].fillna(0)
y = features_df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=50, random_state=42).fit(X_train, y_train)
base_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
print(f"Base honest AUC: {base_auc:.3f} - observed, measured, directional, decision-support")

# TEST 1: label-derived - direct clicks
X_leak1 = X.copy()
X_leak1['leak_label_derived'] = features_df['gsc_clicks']
Xtr1, Xte1, ytr1, yte1 = train_test_split(X_leak1, y, test_size=0.3, random_state=42, stratify=y)
m1 = RandomForestClassifier(n_estimators=50, random_state=42).fit(Xtr1, ytr1)
print(f"Leak Test 1 (label-derived clicks): {roc_auc_score(yte1, m1.predict_proba(Xte1)[:,1]):.3f} -> jumps to perfect")

# TEST 2: future window proxy
X_leak2 = X.copy()
X_leak2['leak_future_impr'] = features_df['gsc_impressions'] * 1.1
Xtr2, Xte2, ytr2, yte2 = train_test_split(X_leak2, y, test_size=0.3, random_state=42, stratify=y)
m2 = RandomForestClassifier(n_estimators=50, random_state=42).fit(Xtr2, ytr2)
print(f"Leak Test 2 (future window proxy): {roc_auc_score(yte2, m2.predict_proba(Xte2)[:,1]):.3f} -> inflated")

# TEST 3: product flag = label
X_leak3 = X.copy()
X_leak3['leak_flag'] = (features_df['gsc_clicks'] > 0).astype(int)
Xtr3, Xte3, ytr3, yte3 = train_test_split(X_leak3, y, test_size=0.3, random_state=42, stratify=y)
m3 = RandomForestClassifier(n_estimators=50, random_state=42).fit(Xtr3, ytr3)
print(f"Leak Test 3 (product flag = label): {roc_auc_score(yte3, m3.predict_proba(Xte3)[:,1]):.3f} -> perfect")

print(f"After deleting all leaks, kept honest AUC: {base_auc:.3f}")

Base honest AUC: 0.929 - observed, measured, directional, decision-support
Leak Test 1 (label-derived clicks): 1.000 -> jumps to perfect
Leak Test 2 (future window proxy): 0.923 -> inflated
Leak Test 3 (product flag = label): 1.000 -> perfect
After deleting all leaks, kept honest AUC: 0.929


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Excluded fields with one-line why each:

- ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other detailed splits - sparse (<1% non-zero), risk of overfitting, not needed for base contract
- scroll_events raw - GA4 event not consistently tracked across clients, use proxy ratio instead
- gsc_sum_position raw - redundant with gsc_avg_position
- sessions_direct, sessions_referral, sessions_social, sessions_paid raw breakdown - use aggregated organic_share and ai_share only to reduce dimensionality
- client_hash_id raw text - high cardinality, re-identification risk per Terms, use frequency encoding only
- content_hash_id raw text - same, use frequency only
- report_date future values or month=2026-06 - sealed test month, never used per assignment
- Any URL, query text, domain - forbidden by privacy, hash only

In [11]:
# Proof excluded not in feature vector
excluded = ['ai_chatgpt','ai_perplexity','ai_gemini','ai_copilot','ai_claude','ai_meta','ai_other','scroll_events','gsc_sum_position']
print("Excluded columns check - not in feature_cols:")
print([c for c in excluded if c not in feature_cols])

# Show sparsity of ai_* to justify exclusion
con.sql("""
SELECT
  SUM(CASE WHEN ai_chatgpt > 0 THEN 1 ELSE 0 END)::DOUBLE / COUNT(*) as chatgpt_nonzero,
  SUM(CASE WHEN sessions_ai > 0 THEN 1 ELSE 0 END)::DOUBLE / COUNT(*) as ai_nonzero
FROM fact
""").df()

Excluded columns check - not in feature_cols:
['ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'gsc_sum_position']


,chatgpt_nonzero,ai_nonzero
0,0.00032,0.0004


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.